# ⚖️ AI-Powered Contract & Legal Document Risk Analyzer
## Teyzix Core Internship | Task AI-3
**Name:** Misbah
**Domain:** Artificial Intelligence
**Difficulty:** Advanced
**Deadline:** 09 July 2026

In [1]:
# ============================================
# CELL 2: Install Libraries
# ============================================

!pip install nltk spacy scikit-learn PyMuPDF \
            python-docx fpdf2 pandas \
            streamlit pyngrok -q

!python -m spacy download en_core_web_sm

print("All libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All libraries installed!


In [2]:
# ============================================
# CELL 3: All Imports
# ============================================

import re
import string
import numpy as np
import pandas as pd
import warnings
import json
import datetime
warnings.filterwarnings('ignore')

# PDF Reading
import fitz  # PyMuPDF

# DOCX Reading
from docx import Document as DocxDocument

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# File Handling
from fpdf import FPDF
import pandas as pd
from google.colab import files

# NLTK Downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# User Database (Simple Dictionary)
USER_DB = {}
DOCUMENT_HISTORY = []
ADMIN_LOGS = []

print("All imports successful!")
print("NLTK data downloaded!")
print("Database initialized!")

All imports successful!
NLTK data downloaded!
Database initialized!


In [3]:
# ============================================
# CELL 4: User Authentication Module
# ============================================

import hashlib

def hash_password(password):
    """Hashes password for secure storage."""
    return hashlib.sha256(password.encode()).hexdigest()


def register_user(username, password, email, role="user"):
    """
    Registers a new user.
    Input : Username, Password, Email, Role
    Output: Success or error message
    """
    if username in USER_DB:
        return False, "Username already exists!"

    USER_DB[username] = {
        'username': username,
        'password': hash_password(password),
        'email': email,
        'role': role,
        'created_at': datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
        'documents': [],
        'last_login': None
    }

    # Log admin action
    ADMIN_LOGS.append({
        'action': 'User Registered',
        'username': username,
        'role': role,
        'time': datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    })

    print(f" User '{username}' registered successfully!")
    return True, "Registration successful!"


def login_user(username, password):
    """
    Logs in existing user.
    Input : Username, Password
    Output: Success or error message
    """
    if username not in USER_DB:
        return False, "Username not found!"

    if USER_DB[username]['password'] != hash_password(password):
        return False, "Incorrect password!"

    # Update last login
    USER_DB[username]['last_login'] = \
        datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    # Log login
    ADMIN_LOGS.append({
        'action': 'User Login',
        'username': username,
        'time': datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    })

    print(f" Welcome back, {username}!")
    print(f"   Role     : {USER_DB[username]['role']}")
    print(f"   Email    : {USER_DB[username]['email']}")
    print(f"   Login At : {USER_DB[username]['last_login']}")
    return True, USER_DB[username]


def show_user_profile(username):
    """Shows user profile information."""
    if username not in USER_DB:
        print("User not found!")
        return

    user = USER_DB[username]
    print("\n" + "="*45)
    print("USER PROFILE")
    print("="*45)
    print(f"Username    : {user['username']}")
    print(f"Email       : {user['email']}")
    print(f"Role        : {user['role']}")
    print(f"Created At  : {user['created_at']}")
    print(f"Last Login  : {user['last_login']}")
    print(f"Documents   : {len(user['documents'])}")
    print("="*45)


def show_admin_panel():
    """Shows admin panel with all users and logs."""
    print("\n" + "="*45)
    print("ADMIN PANEL")
    print("="*45)

    print(f"\nTotal Users: {len(USER_DB)}")
    print("\nUSER LIST:")
    print("-"*45)
    for username, info in USER_DB.items():
        print(f"  {username:<15} | {info['role']:<10} | {info['email']}")

    print(f"\nTotal Logs: {len(ADMIN_LOGS)}")
    print("\nRECENT LOGS:")
    print("-"*45)
    for log in ADMIN_LOGS[-5:]:
        print(f"  {log['time']} | {log['action']} | {log['username']}")

    print("="*45)

print("Authentication Module ready!")
print("   Register function ready")
print("   Login function ready")
print("   Admin panel ready")

Authentication Module ready!
   Register function ready
   Login function ready
   Admin panel ready


In [4]:
# ============================================
# CELL 5: Document Upload & Reader Module
# ============================================

def read_pdf_document(filepath):
    """
    Reads text from PDF document.
    Input : PDF file path
    Output: Extracted text
    """
    try:
        doc = fitz.open(filepath)
        text = ""
        for i, page in enumerate(doc):
            text += page.get_text()
            print(f"  Page {i+1} extracted!")
        print(f"PDF loaded! ({len(doc)} pages)")
        return text
    except Exception as e:
        print(f"PDF Error: {e}")
        return None


def read_docx_document(filepath):
    """
    Reads text from DOCX document.
    Input : DOCX file path
    Output: Extracted text
    """
    try:
        doc = DocxDocument(filepath)
        text = ""
        for para in doc.paragraphs:
            text += para.text + "\n"
        print(f"DOCX loaded! ({len(doc.paragraphs)} paragraphs)")
        return text
    except Exception as e:
        print(f"DOCX Error: {e}")
        return None


def read_txt_document(filepath):
    """
    Reads text from TXT document.
    Input : TXT file path
    Output: Extracted text
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        print(f"TXT loaded! ({len(text)} characters)")
        return text
    except Exception as e:
        print(f"TXT Error: {e}")
        return None


def validate_document(text):
    """
    Validates uploaded document.
    Input : Document text
    Output: True or False
    """
    if not text:
        return False, "Empty document!"
    if len(text.strip()) < 100:
        return False, "Document too short!"
    if len(text) > 1000000:
        return False, "Document too large!"
    return True, "Document valid!"


def upload_document(username):
    """
    Uploads and reads document in Colab.
    Input : Username
    Output: Document text and info
    """
    try:
        print("Upload your document (PDF, DOCX, or TXT):")
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded!")
            return None, None

        filename = list(uploaded.keys())[0]
        print(f"\nProcessing: {filename}")

        # Save file
        with open(filename, 'wb') as f:
            f.write(uploaded[filename])

        # Read based on extension
        if filename.endswith('.pdf'):
            text = read_pdf_document(filename)
        elif filename.endswith('.docx'):
            text = read_docx_document(filename)
        elif filename.endswith('.txt'):
            text = read_txt_document(filename)
        else:
            print("Unsupported format!")
            return None, None

        # Validate document
        is_valid, message = validate_document(text)
        print(message)

        if not is_valid:
            return None, None

        # Create document info
        doc_info = {
            'filename': filename,
            'username': username,
            'upload_time': datetime.datetime.now().strftime(
                "%Y-%m-%d %H:%M"
            ),
            'size': len(text),
            'pages': text.count('\n'),
            'text': text
        }

        # Save to user history
        if username in USER_DB:
            USER_DB[username]['documents'].append(filename)

        # Save to document history
        DOCUMENT_HISTORY.append(doc_info)

        # Log upload
        ADMIN_LOGS.append({
            'action': 'Document Uploaded',
            'username': username,
            'filename': filename,
            'time': datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
        })

        print(f"\n Document ready for analysis!")
        return text, doc_info

    except Exception as e:
        print(f"Upload error: {e}")
        return None, None


def use_sample_document():
    """
    Returns a sample legal contract for testing.
    Output: Sample contract text
    """
    sample_contract = """
    SERVICE AGREEMENT CONTRACT

    This Service Agreement ("Agreement") is entered into as of
    January 1, 2026, between ABC Corporation ("Client") and
    XYZ Services Ltd ("Service Provider").

    PARTIES INVOLVED:
    Client: ABC Corporation, 123 Business Street, Karachi
    Service Provider: XYZ Services Ltd, 456 Tech Avenue, Lahore

    EFFECTIVE DATE: January 1, 2026
    EXPIRY DATE: December 31, 2026

    PAYMENT TERMS:
    The Client agrees to pay Service Provider $5,000 per month.
    Payment must be made within 7 days of invoice.
    Late payments will incur a penalty of 5% per month.
    All payments are non-refundable under any circumstances.

    CONFIDENTIALITY:
    Both parties agree to maintain strict confidentiality.
    No information shall be disclosed to third parties.
    This clause remains effective for 5 years after termination.

    TERMINATION:
    Either party may terminate with 30 days written notice.
    Immediate termination allowed for breach of contract.
    Upon termination all data must be returned immediately.

    RENEWAL:
    This agreement automatically renews annually.
    Either party must give 60 days notice to prevent renewal.

    RESPONSIBILITIES:
    Service Provider must deliver services on time.
    Client must provide necessary access and resources.
    Both parties must comply with applicable laws.

    LIMITATION OF LIABILITY:
    Service Provider liability limited to $1,000 total.
    No liability for indirect or consequential damages.
    Client assumes all risks associated with service use.

    DISPUTE RESOLUTION:
    All disputes resolved through arbitration only.
    Arbitration location: London, United Kingdom.
    Governing law: Laws of United Kingdom apply.

    INTELLECTUAL PROPERTY:
    All work products belong to Service Provider.
    Client receives limited license to use deliverables.
    Client cannot modify or resell any deliverables.

    FORCE MAJEURE:
    Neither party liable for delays due to force majeure.
    Force majeure includes natural disasters and pandemics.

    This agreement constitutes the entire understanding
    between the parties and supersedes all prior agreements.
    """

    print("Sample contract loaded!")
    print(f"   Size: {len(sample_contract)} characters")
    return sample_contract

print("Document Upload & Reader ready!")
print("   PDF reader ready")
print("   DOCX reader ready")
print("   TXT reader ready")
print("   Document validator ready")

Document Upload & Reader ready!
   PDF reader ready
   DOCX reader ready
   TXT reader ready
   Document validator ready


In [5]:
# ============================================
# CELL 6: Contract Analyzer Module
# ============================================

def identify_contract_type(text):
    """
    Identifies type of contract.
    Input : Contract text
    Output: Contract type string
    """
    text_lower = text.lower()

    contract_types = {
        'Service Agreement': [
            'service agreement', 'service contract',
            'service provider', 'services'
        ],
        'Employment Contract': [
            'employment', 'employee', 'employer',
            'salary', 'job', 'work'
        ],
        'Non Disclosure Agreement': [
            'nda', 'non disclosure', 'confidential',
            'trade secret'
        ],
        'Purchase Agreement': [
            'purchase', 'buy', 'sell', 'sale',
            'buyer', 'seller'
        ],
        'Lease Agreement': [
            'lease', 'rent', 'tenant', 'landlord',
            'property'
        ],
        'Partnership Agreement': [
            'partnership', 'partner', 'joint venture',
            'collaboration'
        ],
        'Software License': [
            'software', 'license', 'saas',
            'subscription', 'api'
        ]
    }

    scores = {}
    for contract_type, keywords in contract_types.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        scores[contract_type] = score

    best_type = max(scores, key=scores.get)
    if scores[best_type] == 0:
        return "General Agreement"
    return best_type


def extract_parties(text):
    """
    Extracts parties involved in contract.
    Input : Contract text
    Output: List of parties
    """
    parties = []
    patterns = [
        r'between\s+([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)\s+and\s+([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)',
        r'([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)\s+\("Client"\)',
        r'([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)\s+\("Service Provider"\)',
        r'([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)\s+\("Company"\)',
        r'([A-Z][A-Za-z\s]+(?:Ltd|LLC|Inc|Corp|Co)?)\s+\("Employer"\)',
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            if isinstance(match, tuple):
                parties.extend([m.strip() for m in match])
            else:
                parties.append(match.strip())

    # Remove duplicates
    parties = list(set(parties))
    return parties[:5] if parties else ["Not identified"]


def extract_dates(text):
    """
    Extracts important dates from contract.
    Input : Contract text
    Output: Dictionary of dates
    """
    dates = {
        'effective_date': 'Not found',
        'expiry_date': 'Not found',
        'other_dates': []
    }

    # Date patterns
    date_patterns = [
        r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}',
        r'(?:January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\s+\d{1,2},?\s+\d{4}',
        r'\d{4}-\d{2}-\d{2}'
    ]

    all_dates = []
    for pattern in date_patterns:
        found = re.findall(pattern, text, re.IGNORECASE)
        all_dates.extend(found)

    # Effective date
    effective_patterns = [
        r'effective\s+(?:date\s+)?(?:of\s+)?(.+?\d{4})',
        r'entered\s+into\s+as\s+of\s+(.+?\d{4})',
        r'effective\s+from\s+(.+?\d{4})'
    ]
    for pattern in effective_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            dates['effective_date'] = match.group(1).strip()
            break

    # Expiry date
    expiry_patterns = [
        r'expir(?:y|es|ation)\s+(?:date\s+)?(?:of\s+)?(.+?\d{4})',
        r'terminat(?:es|ion date)\s+(?:on\s+)?(.+?\d{4})',
        r'valid\s+until\s+(.+?\d{4})'
    ]
    for pattern in expiry_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            dates['expiry_date'] = match.group(1).strip()
            break

    dates['other_dates'] = all_dates[:5]
    return dates


def extract_payment_terms(text):
    """
    Extracts payment terms from contract.
    Input : Contract text
    Output: List of payment terms
    """
    payment_keywords = [
        'payment', 'pay', 'fee', 'cost', 'price',
        'amount', 'invoice', 'billing', 'charge',
        'refund', 'penalty', 'late payment'
    ]

    sentences = sent_tokenize(text)
    payment_terms = []

    for sentence in sentences:
        sentence_lower = sentence.lower()
        if any(kw in sentence_lower for kw in payment_keywords):
            clean = sentence.strip()
            if len(clean) > 20:
                payment_terms.append(clean)

    return payment_terms[:5] if payment_terms else ["Not found"]


def extract_clauses(text):
    """
    Extracts important clauses from contract.
    Input : Contract text
    Output: Dictionary of clauses
    """
    clause_types = {
        'confidentiality': [
            'confidential', 'non-disclosure', 'secret',
            'proprietary', 'private'
        ],
        'termination': [
            'terminat', 'cancel', 'end', 'expire',
            'dissolv'
        ],
        'renewal': [
            'renew', 'extend', 'automatic renewal',
            'roll over'
        ],
        'liability': [
            'liabilit', 'liable', 'responsibility',
            'indemnif', 'warrant'
        ],
        'intellectual_property': [
            'intellectual property', 'copyright',
            'trademark', 'patent', 'ownership'
        ],
        'dispute': [
            'dispute', 'arbitration', 'mediation',
            'litigation', 'governing law'
        ]
    }

    sentences = sent_tokenize(text)
    clauses = {}

    for clause_type, keywords in clause_types.items():
        found_sentences = []
        for sentence in sentences:
            sentence_lower = sentence.lower()
            if any(kw in sentence_lower for kw in keywords):
                clean = sentence.strip()
                if len(clean) > 20:
                    found_sentences.append(clean)

        clauses[clause_type] = found_sentences[:3] if found_sentences else ["Not found"]

    return clauses


def analyze_contract(text):
    """
    Complete contract analysis.
    Input : Contract text
    Output: Dictionary with all analysis
    """
    print("\nAnalyzing contract...")
    print("="*55)

    analysis = {
        'contract_type': identify_contract_type(text),
        'parties': extract_parties(text),
        'dates': extract_dates(text),
        'payment_terms': extract_payment_terms(text),
        'clauses': extract_clauses(text),
        'word_count': len(text.split()),
        'sentence_count': len(sent_tokenize(text)),
        'analysis_time': datetime.datetime.now().strftime(
            "%Y-%m-%d %H:%M"
        )
    }

    print(f"  Contract Type  : {analysis['contract_type']}")
    print(f"  Parties Found  : {len(analysis['parties'])}")
    print(f"  Dates Found    : {analysis['dates']['effective_date']}")
    print(f"  Payment Terms  : {len(analysis['payment_terms'])}")
    print(f"  Clauses Found  : {len(analysis['clauses'])}")
    print(f"  Word Count     : {analysis['word_count']}")

    return analysis

print("Contract Analyzer ready!")
print("   Contract type identifier ready")
print("   Parties extractor ready")
print("   Dates extractor ready")
print("   Payment terms extractor ready")
print("   Clauses extractor ready")

Contract Analyzer ready!
   Contract type identifier ready
   Parties extractor ready
   Dates extractor ready
   Payment terms extractor ready
   Clauses extractor ready


In [6]:
# ============================================
# CELL 7: Risk Detector Module
# ============================================

def detect_missing_clauses(text):
    """
    Detects missing important clauses.
    Input : Contract text
    Output: List of missing clauses with confidence
    """
    important_clauses = {
        'Confidentiality Clause': [
            'confidential', 'non-disclosure', 'secret'
        ],
        'Termination Clause': [
            'terminat', 'cancel', 'end agreement'
        ],
        'Payment Terms': [
            'payment', 'pay', 'invoice', 'fee'
        ],
        'Dispute Resolution': [
            'dispute', 'arbitration', 'mediation'
        ],
        'Intellectual Property': [
            'intellectual property', 'copyright', 'ownership'
        ],
        'Limitation of Liability': [
            'liabilit', 'liable', 'limit'
        ],
        'Force Majeure': [
            'force majeure', 'act of god', 'natural disaster'
        ],
        'Governing Law': [
            'governing law', 'jurisdiction', 'applicable law'
        ],
        'Renewal Terms': [
            'renew', 'extend', 'automatic renewal'
        ],
        'Amendment Clause': [
            'amend', 'modify', 'change', 'update'
        ]
    }

    text_lower = text.lower()
    missing_clauses = []

    for clause, keywords in important_clauses.items():
        found = any(kw in text_lower for kw in keywords)
        if not found:
            missing_clauses.append({
                'clause': clause,
                'risk_level': 'High',
                'confidence': 95,
                'explanation': f"{clause} is missing from this contract. This could create legal vulnerabilities."
            })

    return missing_clauses


def detect_high_risk_conditions(text):
    """
    Detects high risk conditions in contract.
    Input : Contract text
    Output: List of high risk conditions
    """
    high_risk_patterns = {
        'Non Refundable Payment': {
            'keywords': ['non-refundable', 'no refund', 'not refundable'],
            'risk_level': 'High',
            'confidence': 90,
            'explanation': 'Contract contains non-refundable payment terms which could result in financial loss.'
        },
        'Automatic Renewal': {
            'keywords': ['automatically renew', 'auto renew', 'automatic renewal'],
            'risk_level': 'Medium',
            'confidence': 85,
            'explanation': 'Contract automatically renews which may result in unexpected charges.'
        },
        'Unlimited Liability': {
            'keywords': ['unlimited liability', 'fully liable', 'responsible for all'],
            'risk_level': 'High',
            'confidence': 92,
            'explanation': 'Contract may expose party to unlimited financial liability.'
        },
        'One Sided Termination': {
            'keywords': ['sole discretion', 'without cause', 'immediate termination'],
            'risk_level': 'High',
            'confidence': 88,
            'explanation': 'Contract allows one sided termination without proper notice.'
        },
        'Late Payment Penalty': {
            'keywords': ['late payment', 'penalty', 'interest on late'],
            'risk_level': 'Medium',
            'confidence': 80,
            'explanation': 'Contract includes late payment penalties which could increase costs.'
        },
        'IP Ownership Risk': {
            'keywords': ['work product belongs', 'intellectual property belongs to provider',
                        'all rights reserved by'],
            'risk_level': 'High',
            'confidence': 87,
            'explanation': 'Contract may transfer intellectual property rights unfavorably.'
        },
        'Arbitration Only': {
            'keywords': ['arbitration only', 'resolved through arbitration'],
            'risk_level': 'Medium',
            'confidence': 75,
            'explanation': 'Contract limits dispute resolution to arbitration only.'
        },
        'Foreign Jurisdiction': {
            'keywords': ['laws of united kingdom', 'laws of usa',
                        'laws of singapore', 'foreign jurisdiction'],
            'risk_level': 'Medium',
            'confidence': 82,
            'explanation': 'Contract governed by foreign jurisdiction which may be inconvenient.'
        }
    }

    text_lower = text.lower()
    detected_risks = []

    for risk_name, risk_info in high_risk_patterns.items():
        found = any(kw in text_lower for kw in risk_info['keywords'])
        if found:
            detected_risks.append({
                'risk': risk_name,
                'risk_level': risk_info['risk_level'],
                'confidence': risk_info['confidence'],
                'explanation': risk_info['explanation']
            })

    return detected_risks


def detect_ambiguous_statements(text):
    """
    Detects ambiguous statements in contract.
    Input : Contract text
    Output: List of ambiguous statements
    """
    ambiguous_keywords = [
        'reasonable', 'appropriate', 'as needed',
        'may', 'might', 'could', 'should consider',
        'at discretion', 'when necessary', 'if required',
        'substantially', 'approximately', 'generally'
    ]

    sentences = sent_tokenize(text)
    ambiguous = []

    for sentence in sentences:
        sentence_lower = sentence.lower()
        found_keywords = [
            kw for kw in ambiguous_keywords
            if kw in sentence_lower
        ]
        if found_keywords:
            ambiguous.append({
                'statement': sentence.strip()[:100] + "...",
                'ambiguous_terms': found_keywords,
                'risk_level': 'Low',
                'confidence': 70,
                'explanation': f"Contains ambiguous terms: {', '.join(found_keywords)}"
            })

    return ambiguous[:5]


def calculate_risk_score(missing_clauses, high_risks, ambiguous):
    """
    Calculates overall risk score.
    Input : Missing clauses, High risks, Ambiguous statements
    Output: Risk score and level
    """
    # Base score
    score = 100

    # Deduct for missing clauses
    score -= len(missing_clauses) * 8

    # Deduct for high risks
    for risk in high_risks:
        if risk['risk_level'] == 'High':
            score -= 10
        elif risk['risk_level'] == 'Medium':
            score -= 5

    # Deduct for ambiguous statements
    score -= len(ambiguous) * 2

    # Keep score between 0 and 100
    score = max(0, min(100, score))

    # Determine risk level
    if score >= 75:
        risk_level = "Low Risk"
        status = "Safe"
    elif score >= 50:
        risk_level = "Medium Risk"
        status = "Review Needed"
    elif score >= 25:
        risk_level = "High Risk"
        status = "Risky"
    else:
        risk_level = "Critical Risk"
        status = "Very Risky"

    return score, risk_level, status


def detect_all_risks(text):
    """
    Runs complete risk detection.
    Input : Contract text
    Output: Complete risk report
    """
    print("\nDetecting risks...")
    print("="*55)

    # Detect all types of risks
    missing_clauses = detect_missing_clauses(text)
    high_risks = detect_high_risk_conditions(text)
    ambiguous = detect_ambiguous_statements(text)

    # Calculate risk score
    risk_score, risk_level, status = calculate_risk_score(
        missing_clauses, high_risks, ambiguous
    )

    risk_report = {
        'risk_score': risk_score,
        'risk_level': risk_level,
        'status': status,
        'missing_clauses': missing_clauses,
        'high_risks': high_risks,
        'ambiguous_statements': ambiguous,
        'total_issues': len(missing_clauses) + len(high_risks) + len(ambiguous)
    }

    print(f"  Risk Score     : {risk_score}/100")
    print(f"  Risk Level     : {risk_level}")
    print(f"  Status         : {status}")
    print(f"  Missing Clauses: {len(missing_clauses)}")
    print(f"  High Risks     : {len(high_risks)}")
    print(f"  Ambiguous      : {len(ambiguous)}")
    print(f"  Total Issues   : {risk_report['total_issues']}")

    return risk_report

print("Risk Detector ready!")
print("   Missing clauses detector ready")
print("   High risk detector ready")
print("   Ambiguous statements detector ready")
print("   Risk score calculator ready")

Risk Detector ready!
   Missing clauses detector ready
   High risk detector ready
   Ambiguous statements detector ready
   Risk score calculator ready


In [7]:
# ============================================
# CELL 8: AI Summary & Semantic Search Module
# ============================================

def generate_executive_summary(text, analysis, risk_report):
    """
    Generates executive summary of contract.
    Input : Contract text, Analysis, Risk report
    Output: Executive summary string
    """
    sentences = sent_tokenize(text)

    # TF-IDF based summarization
    if len(sentences) > 3:
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(sentences)
        scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
        top_indices = sorted(
            np.argsort(scores)[-5:].tolist()
        )
        summary_sentences = [sentences[i] for i in top_indices]
        summary = ' '.join(summary_sentences)
    else:
        summary = text[:500]

    return summary


def generate_key_obligations(text):
    """
    Extracts key obligations from contract.
    Input : Contract text
    Output: List of obligations
    """
    obligation_keywords = [
        'must', 'shall', 'required', 'obligated',
        'responsible', 'agrees to', 'will provide',
        'must ensure', 'is required'
    ]

    sentences = sent_tokenize(text)
    obligations = []

    for sentence in sentences:
        sentence_lower = sentence.lower()
        if any(kw in sentence_lower for kw in obligation_keywords):
            clean = sentence.strip()
            if len(clean) > 20:
                obligations.append(clean)

    return obligations[:8] if obligations else ["No obligations found"]


def generate_recommendations(risk_report, analysis):
    """
    Generates recommendations based on risks.
    Input : Risk report, Analysis
    Output: List of recommendations
    """
    recommendations = []

    # Missing clause recommendations
    for clause in risk_report['missing_clauses']:
        recommendations.append(
            f"ADD {clause['clause']}: This clause is missing and should be added to protect your interests."
        )

    # High risk recommendations
    for risk in risk_report['high_risks']:
        if risk['risk_level'] == 'High':
            recommendations.append(
                f"REVIEW {risk['risk']}: {risk['explanation']}"
            )

    # General recommendations
    if risk_report['risk_score'] < 50:
        recommendations.append(
            "CONSULT LAWYER: This contract has significant risks. Legal review is strongly recommended."
        )

    if risk_report['risk_score'] >= 75:
        recommendations.append(
            "CONTRACT LOOKS GOOD: Basic risks are covered. Standard review recommended."
        )

    return recommendations[:8] if recommendations else [
        "No specific recommendations. Contract appears standard."
    ]


def generate_complete_summary(text, analysis, risk_report):
    """
    Generates complete AI summary report.
    Input : Contract text, Analysis, Risk report
    Output: Complete summary dictionary
    """
    print("\nGenerating AI Summary...")

    summary = {
        'executive_summary': generate_executive_summary(
            text, analysis, risk_report
        ),
        'key_obligations': generate_key_obligations(text),
        'important_dates': analysis['dates'],
        'important_clauses': analysis['clauses'],
        'recommendations': generate_recommendations(
            risk_report, analysis
        ),
        'generated_at': datetime.datetime.now().strftime(
            "%Y-%m-%d %H:%M"
        )
    }

    print(f"  Executive Summary : Generated")
    print(f"  Key Obligations   : {len(summary['key_obligations'])}")
    print(f"  Recommendations   : {len(summary['recommendations'])}")

    return summary


def semantic_search(query, text):
    """
    Searches contract using natural language query.
    Input : Search query, Contract text
    Output: Relevant sentences
    """
    sentences = sent_tokenize(text)

    if not sentences:
        return ["No content found!"]

    try:
        # TF-IDF vectorization
        vectorizer = TfidfVectorizer(stop_words='english')
        all_texts = [query] + sentences
        tfidf_matrix = vectorizer.fit_transform(all_texts)

        # Calculate similarity
        query_vector = tfidf_matrix[0:1]
        sentence_vectors = tfidf_matrix[1:]
        similarities = cosine_similarity(
            query_vector, sentence_vectors
        )[0]

        # Get top results
        top_indices = np.argsort(similarities)[-5:][::-1]
        results = []

        for idx in top_indices:
            if similarities[idx] > 0.1:
                results.append({
                    'sentence': sentences[idx].strip(),
                    'relevance': round(similarities[idx] * 100, 2)
                })

        return results if results else [
            {"sentence": "No relevant content found!", "relevance": 0}
        ]

    except Exception as e:
        return [{"sentence": f"Search error: {e}", "relevance": 0}]


def show_search_results(query, results):
    """
    Displays search results in formatted way.
    Input : Query, Results list
    Output: Prints formatted results
    """
    print("\n" + "="*55)
    print(f"SEARCH RESULTS FOR: '{query}'")
    print("="*55)

    for i, result in enumerate(results, 1):
        print(f"\nResult {i}:")
        print(f"Relevance : {result['relevance']}%")
        print(f"Content   : {result['sentence'][:150]}...")
        print("-"*45)

print("AI Summary & Search ready!")
print("   Executive summary generator ready")
print("   Key obligations extractor ready")
print("   Recommendations generator ready")
print("   Semantic search ready")

AI Summary & Search ready!
   Executive summary generator ready
   Key obligations extractor ready
   Recommendations generator ready
   Semantic search ready


In [11]:
# ============================================
# CELL 9: Dashboard & Report Generator
# ============================================

def show_dashboard():
    print("\n" + "="*55)
    print("AI INSIGHTS DASHBOARD")
    print("="*55)
    total_docs = len(DOCUMENT_HISTORY)
    print(f"\nTOTAL DOCUMENTS : {total_docs}")
    print(f"TOTAL USERS     : {len(USER_DB)}")
    print(f"TOTAL LOGS      : {len(ADMIN_LOGS)}")
    if total_docs == 0:
        print("\nNo documents processed yet!")
        return
    print("\nDOCUMENT HISTORY:")
    print("-"*45)
    for doc in DOCUMENT_HISTORY:
        print(f"  File    : {doc['filename']}")
        print(f"  User    : {doc['username']}")
        print(f"  Time    : {doc['upload_time']}")
    print("\nRECENT ACTIVITY:")
    print("-"*45)
    for log in ADMIN_LOGS[-5:]:
        print(f"  {log.get('time')} | {log.get('action')} | {log.get('username')}")
    print("="*55)


def show_risk_dashboard(risk_report):
    print("\n" + "="*55)
    print("RISK ANALYSIS DASHBOARD")
    print("="*55)
    score = risk_report['risk_score']
    bar_len = int(score / 5)
    bar = "█" * bar_len
    empty = "░" * (20 - bar_len)
    print(f"\nRISK SCORE: {score}/100")
    print(f"|{bar}{empty}| {risk_report['status']}")
    print(f"RISK LEVEL: {risk_report['risk_level']}")
    print(f"\nMISSING CLAUSES ({len(risk_report['missing_clauses'])}):")
    print("-"*45)
    for clause in risk_report['missing_clauses']:
        print(f"  X {clause['clause']}")
        print(f"    Risk: {clause['risk_level']}")
    print(f"\nDETECTED RISKS ({len(risk_report['high_risks'])}):")
    print("-"*45)
    for risk in risk_report['high_risks']:
        print(f"  ! {risk['risk']}")
        print(f"    Level: {risk['risk_level']}")
    print(f"\nAMBIGUOUS ({len(risk_report['ambiguous_statements'])}):")
    print("-"*45)
    for amb in risk_report['ambiguous_statements'][:3]:
        print(f"  ? {amb['statement'][:60]}...")
    print("="*55)


def export_report_txt(analysis, risk_report, summary, doc_info):
    try:
        filename = "Contract_Risk_Report.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("="*50 + "\n")
            f.write("AI CONTRACT RISK ANALYZER\n")
            f.write("Teyzix Core Internship | Task AI-3\n")
            f.write("="*50 + "\n\n")
            f.write("DOCUMENT INFO:\n")
            f.write(f"File: {doc_info.get('filename', 'N/A')}\n")
            f.write(f"Time: {analysis['analysis_time']}\n")
            f.write(f"Words: {analysis['word_count']}\n\n")
            f.write("CONTRACT ANALYSIS:\n")
            f.write(f"Type: {analysis['contract_type']}\n")
            f.write(f"Parties: {', '.join(analysis['parties'])}\n")
            f.write(f"Effective: {analysis['dates']['effective_date']}\n")
            f.write(f"Expiry: {analysis['dates']['expiry_date']}\n\n")
            f.write("RISK ASSESSMENT:\n")
            f.write(f"Score: {risk_report['risk_score']}/100\n")
            f.write(f"Level: {risk_report['risk_level']}\n")
            f.write(f"Status: {risk_report['status']}\n")
            f.write(f"Issues: {risk_report['total_issues']}\n\n")
            f.write("MISSING CLAUSES:\n")
            for clause in risk_report['missing_clauses']:
                f.write(f"- {clause['clause']}\n")
            f.write("\nDETECTED RISKS:\n")
            for risk in risk_report['high_risks']:
                f.write(f"- {risk['risk']} [{risk['risk_level']}]\n")
            f.write("\nRECOMMENDATIONS:\n")
            for i, rec in enumerate(summary['recommendations'], 1):
                f.write(f"{i}. {rec}\n")
        files.download(filename)
        print("✅ TXT Report downloaded!")
        return filename
    except Exception as e:
        print(f"❌ TXT export error: {e}")
        return None


def export_report_pdf(analysis, risk_report, summary, doc_info):
    try:
        pdf = FPDF('P', 'mm', 'A4')
        pdf.add_page()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.set_left_margin(20)
        pdf.set_right_margin(20)
        pdf.set_top_margin(20)
        w = pdf.w - 40

        # Title
        pdf.set_font("Arial", 'B', 13)
        pdf.set_text_color(0, 100, 0)
        pdf.cell(w, 10, "AI Contract Risk Analyzer",
                 ln=True, align='C')

        pdf.set_font("Arial", size=10)
        pdf.set_text_color(80, 80, 80)
        pdf.cell(w, 7, "Teyzix Core Internship | Task AI-3",
                 ln=True, align='C')
        pdf.ln(4)

        # Document Info
        pdf.set_text_color(0, 0, 0)
        pdf.set_fill_color(230, 255, 230)
        pdf.set_font("Arial", 'B', 11)
        pdf.cell(w, 8, "Document Information",
                 ln=True, fill=True)
        pdf.set_font("Arial", size=9)
        pdf.cell(w, 6,
            f"Contract Type: {str(analysis['contract_type'])[:40]}",
            ln=True)
        pdf.cell(w, 6,
            f"Word Count: {analysis['word_count']}",
            ln=True)
        pdf.cell(w, 6,
            f"Analyzed: {analysis['analysis_time']}",
            ln=True)
        pdf.ln(3)

        # Risk Score
        pdf.set_fill_color(255, 230, 230)
        pdf.set_font("Arial", 'B', 11)
        pdf.cell(w, 8, "Risk Assessment", ln=True, fill=True)
        pdf.set_font("Arial", size=9)
        pdf.cell(w, 6,
            f"Score: {risk_report['risk_score']}/100",
            ln=True)
        pdf.cell(w, 6,
            f"Level: {risk_report['risk_level']}",
            ln=True)
        pdf.cell(w, 6,
            f"Status: {risk_report['status']}",
            ln=True)
        pdf.cell(w, 6,
            f"Total Issues: {risk_report['total_issues']}",
            ln=True)
        pdf.ln(3)

        # Missing Clauses
        pdf.set_fill_color(255, 255, 200)
        pdf.set_font("Arial", 'B', 11)
        pdf.cell(w, 8, "Missing Clauses", ln=True, fill=True)
        pdf.set_font("Arial", size=9)
        for clause in risk_report['missing_clauses']:
            line = f"- {clause['clause']}"
            line = line[:55]
            pdf.cell(w, 6, line, ln=True)
        pdf.ln(3)

        # Detected Risks
        pdf.set_fill_color(255, 200, 200)
        pdf.set_font("Arial", 'B', 11)
        pdf.cell(w, 8, "Detected Risks", ln=True, fill=True)
        pdf.set_font("Arial", size=9)
        for risk in risk_report['high_risks']:
            line = f"- {risk['risk']} [{risk['risk_level']}]"
            line = line[:55]
            pdf.cell(w, 6, line, ln=True)
        pdf.ln(3)

        # Recommendations
        pdf.set_fill_color(230, 230, 255)
        pdf.set_font("Arial", 'B', 11)
        pdf.cell(w, 8, "Recommendations", ln=True, fill=True)
        pdf.set_font("Arial", size=9)
        for i, rec in enumerate(
            summary['recommendations'][:5], 1
        ):
            safe = str(rec)[:70]
            safe = safe.encode(
                'latin-1', 'replace'
            ).decode('latin-1')
            pdf.multi_cell(w, 6, f"{i}. {safe}")
            pdf.ln(1)

        filename = "Contract_Risk_Report.pdf"
        pdf.output(filename)
        files.download(filename)
        print("✅ PDF Report downloaded!")
        return filename

    except Exception as e:
        print(f"❌ PDF export error: {e}")
        return None

print("Dashboard & Report Generator ready!")
print("   AI Dashboard ready")
print("   Risk Dashboard ready")
print("   TXT Report generator ready")
print("   PDF Report generator ready")

Dashboard & Report Generator ready!
   AI Dashboard ready
   Risk Dashboard ready
   TXT Report generator ready
   PDF Report generator ready


In [12]:
# ============================================
# CELL 10: MAIN PROGRAM
# ============================================

def main():
    print("="*55)
    print("AI CONTRACT & LEGAL DOCUMENT RISK ANALYZER")
    print("Teyzix Core Internship | Task AI-3")
    print("="*55)

    # STEP 1: AUTHENTICATION
    print("\nSTEP 1 - USER AUTHENTICATION:")
    print("  1. Register New Account")
    print("  2. Login Existing Account")
    print("  3. Continue as Guest")
    auth_choice = input("\nEnter Choice (1/2/3): ").strip()

    username = "guest"

    if auth_choice == "1":
        print("\nREGISTER NEW ACCOUNT:")
        username = input("Enter Username: ").strip()
        password = input("Enter Password: ").strip()
        email = input("Enter Email: ").strip()
        success, message = register_user(
            username, password, email
        )
        if not success:
            print(message)
            return

        # Auto login after register
        success, user_info = login_user(username, password)

    elif auth_choice == "2":
        print("\nLOGIN:")
        username = input("Enter Username: ").strip()
        password = input("Enter Password: ").strip()
        success, message = login_user(username, password)
        if not success:
            print(message)
            return

    else:
        print("\nContinuing as Guest...")
        username = "guest"

    # STEP 2: DOCUMENT INPUT
    print("\nSTEP 2 - DOCUMENT INPUT:")
    print("  1. Upload Document (PDF/DOCX/TXT)")
    print("  2. Use Sample Contract")
    doc_choice = input("\nEnter Choice (1/2): ").strip()

    if doc_choice == "1":
        text, doc_info = upload_document(username)
        if not text:
            print("❌ Document loading failed!")
            return
    else:
        text = use_sample_document()
        doc_info = {
            'filename': 'sample_contract.txt',
            'username': username,
            'upload_time': datetime.datetime.now().strftime(
                "%Y-%m-%d %H:%M"
            ),
            'size': len(text)
        }
        DOCUMENT_HISTORY.append(doc_info)

    # STEP 3: CONTRACT ANALYSIS
    print("\nSTEP 3 - ANALYZING CONTRACT...")
    analysis = analyze_contract(text)

    # STEP 4: RISK DETECTION
    print("\nSTEP 4 - DETECTING RISKS...")
    risk_report = detect_all_risks(text)

    # STEP 5: AI SUMMARY
    print("\nSTEP 5 - GENERATING AI SUMMARY...")
    summary = generate_complete_summary(
        text, analysis, risk_report
    )

    # STEP 6: SHOW RESULTS
    print("\nSTEP 6 - RESULTS:")

    # Contract Analysis
    print("\n" + "="*55)
    print("CONTRACT ANALYSIS RESULTS")
    print("="*55)
    print(f"Contract Type  : {analysis['contract_type']}")
    print(f"Parties        : {', '.join(analysis['parties'])}")
    print(f"Effective Date : {analysis['dates']['effective_date']}")
    print(f"Expiry Date    : {analysis['dates']['expiry_date']}")
    print(f"Word Count     : {analysis['word_count']}")

    # Payment Terms
    print("\nPAYMENT TERMS:")
    for term in analysis['payment_terms'][:3]:
        print(f"  → {term[:80]}...")

    # Risk Dashboard
    show_risk_dashboard(risk_report)

    # Executive Summary
    print("\n" + "="*55)
    print("EXECUTIVE SUMMARY:")
    print("="*55)
    print(summary['executive_summary'][:400] + "...")

    # Key Obligations
    print("\nKEY OBLIGATIONS:")
    print("-"*45)
    for i, obligation in enumerate(
        summary['key_obligations'][:5], 1
    ):
        print(f"  {i}. {obligation[:80]}...")

    # Recommendations
    print("\nRECOMMENDATIONS:")
    print("-"*45)
    for i, rec in enumerate(summary['recommendations'], 1):
        print(f"  {i}. {rec}")

    # STEP 7: SEMANTIC SEARCH
    print("\nSTEP 7 - SEMANTIC SEARCH:")
    print("  1. Search payment terms")
    print("  2. Search confidentiality clauses")
    print("  3. Search termination conditions")
    print("  4. Custom search")
    print("  5. Skip search")
    search_choice = input("\nEnter Choice (1/2/3/4/5): ").strip()

    search_queries = {
        "1": "payment terms fee invoice",
        "2": "confidentiality secret disclosure",
        "3": "termination cancel end agreement",
    }

    if search_choice in search_queries:
        query = search_queries[search_choice]
        results = semantic_search(query, text)
        show_search_results(query, results)
    elif search_choice == "4":
        query = input("Enter your search query: ").strip()
        results = semantic_search(query, text)
        show_search_results(query, results)

    # STEP 8: DASHBOARD
    print("\nSTEP 8 - DASHBOARD:")
    show_dashboard()

    # STEP 9: EXPORT REPORT
    print("\nSTEP 9 - EXPORT REPORT:")
    print("  1. Save as TXT")
    print("  2. Save as PDF")
    print("  3. Save Both")
    print("  4. Skip")
    export = input("\nEnter Choice (1/2/3/4): ").strip()

    if export == "1":
        export_report_txt(analysis, risk_report, summary, doc_info)
    elif export == "2":
        export_report_pdf(analysis, risk_report, summary, doc_info)
    elif export == "3":
        export_report_txt(analysis, risk_report, summary, doc_info)
        export_report_pdf(analysis, risk_report, summary, doc_info)
    else:
        print("Export skipped!")

    # STEP 10: ADMIN PANEL
    if username in USER_DB and \
       USER_DB[username]['role'] == 'admin':
        print("\nSTEP 10 - ADMIN PANEL:")
        show_admin_panel()

    print("\n" + "="*55)
    print("TASK COMPLETE!")
    print(f"Risk Score : {risk_report['risk_score']}/100")
    print(f"Risk Level : {risk_report['risk_level']}")
    print(f"Status     : {risk_report['status']}")
    print("="*55)

main()

AI CONTRACT & LEGAL DOCUMENT RISK ANALYZER
Teyzix Core Internship | Task AI-3

STEP 1 - USER AUTHENTICATION:
  1. Register New Account
  2. Login Existing Account
  3. Continue as Guest

Enter Choice (1/2/3): 2

LOGIN:
Enter Username: misbah
Enter Password: misbah123
 Welcome back, misbah!
   Role     : user
   Email    : misbah@email.com
   Login At : 2026-07-07 07:59

STEP 2 - DOCUMENT INPUT:
  1. Upload Document (PDF/DOCX/TXT)
  2. Use Sample Contract

Enter Choice (1/2): 2
Sample contract loaded!
   Size: 2198 characters

STEP 3 - ANALYZING CONTRACT...

Analyzing contract...
  Contract Type  : Service Agreement
  Parties Found  : 2
  Dates Found    : DATE: January 1, 2026
  Payment Terms  : 4
  Clauses Found  : 6
  Word Count     : 279

STEP 4 - DETECTING RISKS...

Detecting risks...
  Risk Score     : 58/100
  Risk Level     : Medium Risk
  Status         : Review Needed
  Missing Clauses: 0
  High Risks     : 6
  Ambiguous      : 1
  Total Issues   : 7

STEP 5 - GENERATING AI SUM

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ TXT Report downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ PDF Report downloaded!

TASK COMPLETE!
Risk Score : 58/100
Risk Level : Medium Risk
Status     : Review Needed


In [13]:
# ============================================
# CELL 11: Quick Test
# ============================================

print("QUICK TEST RUNNING...")
print("="*55)

# Test Authentication
print("\nTesting Authentication...")
register_user("testuser", "test123", "test@email.com")
success, info = login_user("testuser", "test123")

# Test Document
print("\nTesting Document Reader...")
text = use_sample_document()

# Test Analysis
print("\nTesting Contract Analyzer...")
analysis = analyze_contract(text)

# Test Risk Detection
print("\nTesting Risk Detector...")
risk_report = detect_all_risks(text)

# Test Summary
print("\nTesting AI Summary...")
summary = generate_complete_summary(
    text, analysis, risk_report
)

# Test Search
print("\nTesting Semantic Search...")
results = semantic_search("payment terms", text)
show_search_results("payment terms", results)

# Test Dashboard
print("\nTesting Dashboard...")
show_risk_dashboard(risk_report)

print("\n" + "="*55)
print("ALL TESTS PASSED!")
print(f"Risk Score : {risk_report['risk_score']}/100")
print(f"Risk Level : {risk_report['risk_level']}")
print(f"Status     : {risk_report['status']}")
print("="*55)

QUICK TEST RUNNING...

Testing Authentication...
 User 'testuser' registered successfully!
 Welcome back, testuser!
   Role     : user
   Email    : test@email.com
   Login At : 2026-07-07 08:01

Testing Document Reader...
Sample contract loaded!
   Size: 2198 characters

Testing Contract Analyzer...

Analyzing contract...
  Contract Type  : Service Agreement
  Parties Found  : 2
  Dates Found    : DATE: January 1, 2026
  Payment Terms  : 4
  Clauses Found  : 6
  Word Count     : 279

Testing Risk Detector...

Detecting risks...
  Risk Score     : 58/100
  Risk Level     : Medium Risk
  Status         : Review Needed
  Missing Clauses: 0
  High Risks     : 6
  Ambiguous      : 1
  Total Issues   : 7

Testing AI Summary...

Generating AI Summary...
  Executive Summary : Generated
  Key Obligations   : 8
  Recommendations   : 2

Testing Semantic Search...

SEARCH RESULTS FOR: 'payment terms'

Result 1:
Relevance : 35.97%
Content   : Payment must be made within 7 days of invoice....
-----

# README

# AI-Powered Contract & Legal Document Risk Analyzer

## Task Information
| Field | Detail |
|-------|--------|
| Task ID | AI-3 |
| Name | Misbah |
| Domain | Artificial Intelligence |
| Difficulty | Advanced |
| Deadline | 09 July 2026 |

## Objective
Automatically analyze legal documents, identify important
clauses, detect potential risks, summarize documents,
and generate actionable insights using AI.

## Features
- User Registration and Login
- Admin Panel
- PDF, DOCX, TXT document upload
- Contract type identification
- Parties extraction
- Dates extraction
- Payment terms extraction
- Clause extraction
- Risk detection with confidence scores
- Missing clauses detection
- Ambiguous statements detection
- Risk score calculation
- AI executive summary
- Key obligations extraction
- Recommendations generation
- Semantic search
- AI insights dashboard
- TXT and PDF report export
- Document history

## Libraries Used
- nltk
- spacy
- scikit-learn
- PyMuPDF
- python-docx
- pandas
- fpdf2

## Risk Scoring System
| Score | Level | Status |
|-------|-------|--------|
| 75-100 | Low Risk | Safe |
| 50-74 | Medium Risk | Review Needed |
| 25-49 | High Risk | Risky |
| 0-24 | Critical Risk | Very Risky |

## How To Run
1. Open Google Colab
2. Run Cell 2 - Install libraries
3. Run Cell 3 - Import libraries
4. Run Cell 4 to 9 - Load all modules
5. Run Cell 10 - Main program
6. Register or login
7. Upload document or use sample
8. Get complete risk analysis!

## Deliverables
- Complete source code
- GitHub Repository
- Sample legal documents
- Risk reports (TXT and PDF)
- Screenshots
- README documentation

In [14]:
# ============================================
# CELL 13: Create README.md File
# ============================================

readme_content = """# README

# AI-Powered Contract & Legal Document Risk Analyzer

## Task Information
- Task ID: AI-3
- Name: Misbah
- Domain: Artificial Intelligence
- Difficulty: Advanced
- Deadline: 09 July 2026

## Objective
Automatically analyze legal documents, identify important
clauses, detect potential risks, summarize documents,
and generate actionable insights using AI.

## Features
- User Registration and Login
- Admin Panel
- PDF DOCX TXT document upload
- Contract type identification
- Parties extraction
- Risk detection with confidence scores
- Missing clauses detection
- Risk score calculation
- AI executive summary
- Semantic search
- TXT and PDF report export
- Document history

## Libraries Used
- nltk
- spacy
- scikit-learn
- PyMuPDF
- python-docx
- pandas
- fpdf2

## Risk Scoring System
- 75-100: Low Risk - Safe
- 50-74 : Medium Risk - Review Needed
- 25-49 : High Risk - Risky
- 0-24  : Critical Risk - Very Risky

## How To Run
1. Open Google Colab
2. Run Cell 2 - Install libraries
3. Run Cell 3 - Imports
4. Run Cell 4 to 9 - Load modules
5. Run Cell 10 - Main program
6. Register or login
7. Upload document or use sample
8. Get complete risk analysis!
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

files.download('README.md')
print("README.md downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

README.md downloaded!
